# 3 · Cross-isoform fate of Pfam domains (intact / skipped / trimmed)

Reproduces the outcome bar showing **~76 % intact, ~15 % skipped, ~9 % trimmed**.

Starts from the **raw** file `domain_isoform_master.tsv` (one row per
domain-isoform comparison) and pools every comparison.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# raw table (built by scripts 01-02)
MASTER = Path('/data1/peerd/sotougl/fastcds/data/raw/domain_isoform_master.tsv')

In [ ]:
# read only the 3 columns we need from the 505k-row master
m = pd.read_csv(MASTER, sep='\t',
                usecols=['domain_bp', 'covered_bp', 'conservation_class'])

# coverage = domain coding bp retained in the target isoform / domain coding bp
cover = (m.covered_bp / m.domain_bp.replace(0, np.nan)).clip(0, 1)

# one of three outcomes per comparison
outcome = np.where(m.conservation_class == 'conserved', 'intact',
          np.where(cover < 0.15, 'skipped', 'trimmed'))

pct = (pd.Series(outcome).value_counts(normalize=True)
         .reindex(['intact', 'trimmed', 'skipped']) * 100)
print(f'{len(m):,} domain-isoform comparisons')
print(pct.round(1))

In [ ]:
# horizontal stacked bar: intact (green) / trimmed (orange) / skipped (red)
colors = {'intact': '#1a9850', 'trimmed': '#fdae61', 'skipped': '#d73027'}
order  = ['intact', 'trimmed', 'skipped']

fig, ax = plt.subplots(figsize=(8, 1.9))
left = 0
for k in order:
    ax.barh(0, pct[k], left=left, color=colors[k], edgecolor='white')
    ax.text(left + pct[k] / 2, 0, f'{k}\n{pct[k]:.0f}%', ha='center', va='center',
            color='white', fontsize=12, fontweight='bold')
    left += pct[k]
ax.set_xlim(0, 100); ax.set_ylim(-.5, .5); ax.set_yticks([])
ax.set_xlabel('% of all domain-isoform comparisons')
ax.set_title(f'Per comparison, a domain is kept whole or removed  (n = {len(m):,})')
fig.tight_layout()
plt.show()